# Exploración detallada de la tabla MOSAIC

## Objetivo del notebook

La tabla `mosaic` es la pieza clave para la línea de geomarketing del TFG. A diferencia de las demás tablas del proyecto, no procede del ERP de Selmark sino de un proveedor externo de datos sociodemográficos (Experian Mosaic). Su estructura es compleja (76 columnas) y combina información geográfica, segmentación poblacional y variables socioeconómicas.

Este notebook tiene como propósito comprender en profundidad la estructura, calidad y cobertura de esta tabla, con el fin de tomar decisiones informadas sobre su tratamiento en la capa Silver y su incorporación posterior a la tabla cliente final.

El alcance del notebook incluye:

1. Contexto y descripción general del dato MOSAIC.
2. Clasificación funcional de las 76 columnas en bloques temáticos.
3. Análisis de la clave geográfica (código postal) y su normalización.
4. Distribución de los grupos y segmentos MOSAIC en España.
5. Calidad de las variables socioeconómicas (renta media, conteos).
6. Estimación de cobertura al cruzar con la base de clientes de Selmark.
7. Decisiones de modelado para la capa Silver.

## 1. Contexto del dato MOSAIC

**MOSAIC** es una herramienta de segmentación sociodemográfica desarrollada por Experian, ampliamente utilizada en marketing y geomarketing. Asigna a cada código postal de España un perfil estadístico basado en variables como nivel de renta, estructura familiar, edad, hábitos de consumo y entorno residencial.

La segmentación se organiza en dos niveles:

- **Nivel grupo**: 11 grupos identificados con letras (A a K, más una categoría residual `U`). Cada grupo agrupa perfiles con características socioeconómicas similares.
- **Nivel segmento fino**: aproximadamente 50 segmentos numerados dentro de cada grupo (A1, A2, B5, B6, etc.), que permiten un análisis más granular.

Para cada código postal, la tabla recoge la **proporción de hogares** que pertenecen a cada segmento y a cada grupo, además de identificar el segmento dominante y la renta media estimada.

Esta información permitirá, en fases posteriores del TFG:

- Caracterizar el perfil sociodemográfico predominante de los clientes de Selmark.
- Identificar zonas geográficas con potencial comercial no captado.
- Construir variables de geomarketing para los modelos de clustering y segmentación.

## 2. Configuración del entorno

Se establece la conexión con la base DuckDB en modo de solo lectura.

In [10]:
import duckdb
import pandas as pd
from pathlib import Path

RUTA_PROYECTO = Path("..").resolve()
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)
print(f"Conexión establecida con: {RUTA_DUCKDB}")

Conexión establecida con: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb


## 3. Volumen y dimensiones generales

Se confirma el volumen de la tabla y se obtiene una primera muestra para visualizar el formato de los datos.

In [11]:
# Volumen general
volumen = con.execute("""
    SELECT 
        COUNT(*) AS filas,
        COUNT(DISTINCT CP) AS cps_unicos,
        COUNT(DISTINCT PROV_INE) AS provincias_unicas
    FROM bronze.mosaic
""").fetchdf()

print("Volumen de bronze.mosaic:")
print(volumen.to_string(index=False))

Volumen de bronze.mosaic:
 filas  cps_unicos  provincias_unicas
  6457        6457                 52


## 4. Clasificación funcional de las columnas

Las 76 columnas de la tabla pueden agruparse en bloques temáticos para facilitar su comprensión y posterior tratamiento. A continuación se presenta esta clasificación.

| Bloque | Columnas | Descripción |
|---|---|---|
| **Identificación geográfica** | `CP`, `CP_value`, `PROV`, `PROV_INE` | Código postal y provincia |
| **Segmentos finos (A1 a K50)** | 50 columnas (`A1`, `A2`, `B5`, `B6`, etc.) | Proporción de cada segmento detallado en el CP |
| **Categoría residual** | `U` | Población sin clasificar |
| **Grupos agregados (A a K)** | 11 columnas (`A`, `B`, `C`, ..., `K`) | Proporción agregada por grupo MOSAIC |
| **Categoría residual agregada** | `U2` | Equivalente agregado de `U` |
| **Segmento dominante** | `Max_Mosaic`, `Max_Mosaic1` | Segmento fino mayoritario y su peso |
| **Grupo dominante** | `Max_Mosaic_G`, `Max_Mosaic2` | Grupo (letra) mayoritario y su peso |
| **Variables socioeconómicas** | `Renta_Media` | Renta media estimada del CP |
| **Variables de control** | `F2`, `Count`, `Mosaic_number`, `Check` | Indicadores de calidad y validación |

El análisis posterior se centrará principalmente en los **grupos agregados** (A-K) y en `Max_Mosaic_G`, por ser el nivel más interpretable para análisis comerciales. Los segmentos finos quedarán disponibles para análisis avanzados.

## 5. Análisis del código postal como clave geográfica

El código postal (`CP`) es la clave que permitirá unir esta tabla con `dim_cliente`. Es necesario analizar su formato y consistencia para garantizar que el JOIN posterior funcione correctamente.

In [12]:
# Análisis del formato del CP
analisis_cp = con.execute("""
    SELECT 
        COUNT(*) AS total_cps,
        COUNT(DISTINCT CP) AS cps_distintos,
        COUNT(*) FILTER (WHERE CP IS NULL) AS cps_nulos,
        COUNT(*) FILTER (WHERE LENGTH(CP) = 4) AS cps_long_4,
        COUNT(*) FILTER (WHERE LENGTH(CP) = 5) AS cps_long_5,
        COUNT(*) FILTER (WHERE LENGTH(CP) NOT IN (4, 5)) AS cps_long_otra,
        MIN(CAST(CP AS INTEGER)) AS cp_min,
        MAX(CAST(CP AS INTEGER)) AS cp_max
    FROM bronze.mosaic
""").fetchdf()

print("Análisis del formato del código postal:\n")
print(analisis_cp.to_string(index=False))

Análisis del formato del código postal:

 total_cps  cps_distintos  cps_nulos  cps_long_4  cps_long_5  cps_long_otra  cp_min  cp_max
      6457           6457          0        1243        5214              0    1000   52006


### Hallazgo sobre el formato del CP

Los códigos postales se han cargado **sin ceros iniciales**: por ejemplo, el CP de Álava se almacena como `1000` en lugar de `01000`. Esto se debe a que el archivo CSV original ya venía con esta convención y DuckDB lo respetó.

Sin embargo, en `dim_cliente` los códigos postales **sí conservan los cinco dígitos completos**, dado que esa tabla se cargó forzando todas las columnas a tipo VARCHAR.

**Implicación**: en la capa Silver será imprescindible aplicar padding a 5 dígitos al CP de la tabla MOSAIC antes de realizar el JOIN, para garantizar la coincidencia con `dim_cliente`.

## 6. Distribución geográfica por provincia

A continuación se analiza cuántos códigos postales hay en cada provincia, con el fin de detectar posibles desequilibrios en la cobertura territorial.

In [13]:
# Distribución por provincia (top 15)
distribucion_prov = con.execute("""
    SELECT 
        PROV_INE AS provincia,
        COUNT(*) AS num_cps
    FROM bronze.mosaic
    GROUP BY PROV_INE
    ORDER BY num_cps DESC
    LIMIT 15
""").fetchdf()

print("Top 15 provincias por número de códigos postales:\n")
print(distribucion_prov.to_string(index=False))

# Total de provincias
total_prov = con.execute("SELECT COUNT(DISTINCT PROV_INE) FROM bronze.mosaic").fetchone()[0]
print(f"\nTotal de provincias cubiertas: {total_prov}")

Top 15 provincias por número de códigos postales:

        provincia  num_cps
        Barcelona      376
           Madrid      288
Valencia/València      244
         Asturias      208
        Coruña, A      199
       Pontevedra      192
           Lleida      184
           Girona      181
 Alicante/Alacant      176
          Navarra      169
        Tarragona      163
             León      158
          Granada      154
        Cantabria      151
         Zaragoza      150

Total de provincias cubiertas: 52


## 7. Distribución de los grupos MOSAIC

Se analiza la frecuencia con que cada grupo (A-K) aparece como dominante en los códigos postales españoles. Esta distribución refleja la composición sociodemográfica del territorio nacional según la metodología MOSAIC.

In [14]:
# Distribución del grupo dominante
distribucion_grupos = con.execute("""
    SELECT 
        Max_Mosaic_G AS grupo_dominante,
        COUNT(*) AS num_cps,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM bronze.mosaic
    WHERE Max_Mosaic_G IS NOT NULL
    GROUP BY Max_Mosaic_G
    ORDER BY num_cps DESC
""").fetchdf()

print("Distribución del grupo MOSAIC dominante:\n")
print(distribucion_grupos.to_string(index=False))

Distribución del grupo MOSAIC dominante:

grupo_dominante  num_cps  porcentaje
              K     1971       30.53
              H     1452       22.49
              C      694       10.75
              J      507        7.85
              I      500        7.74
              A      295        4.57
              D      236        3.65
              B      225        3.48
              G      224        3.47
              E      182        2.82
              F      128        1.98
              U       43        0.67


## 8. Distribución del segmento fino dominante

Adicionalmente, se inspecciona la distribución de los segmentos finos dominantes, lo que permite identificar los perfiles más comunes a nivel detallado.

In [15]:
# Top 15 segmentos finos dominantes
distribucion_seg = con.execute("""
    SELECT 
        Max_Mosaic AS segmento_dominante,
        COUNT(*) AS num_cps,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM bronze.mosaic
    WHERE Max_Mosaic IS NOT NULL
    GROUP BY Max_Mosaic
    ORDER BY num_cps DESC
    LIMIT 15
""").fetchdf()

print("Top 15 segmentos finos dominantes:\n")
print(distribucion_seg.to_string(index=False))

Top 15 segmentos finos dominantes:

segmento_dominante  num_cps  porcentaje
               H34      622        9.63
               K49      522        8.08
               K45      406        6.29
               C13      357        5.53
               K47      323        5.00
               I38      253        3.92
               J42      253        3.92
               H33      249        3.86
               H35      232        3.59
               K46      228        3.53
               K50      222        3.44
               J43      212        3.28
               K48      197        3.05
               H31      189        2.93
               C14      169        2.62


## 9. Análisis de la variable `Renta_Media`

La renta media es una de las variables socioeconómicas más relevantes para el análisis territorial. Se evalúa su calidad (presencia de nulos), su rango y su distribución.

In [16]:
# Análisis de Renta_Media
# Nota: la columna está como VARCHAR. Se convierte a numérico para análisis,
# tratando 'NaN' y cadenas no numéricas como nulos.
analisis_renta = con.execute("""
    SELECT 
        COUNT(*) AS total_cps,
        COUNT(*) FILTER (WHERE Renta_Media IS NULL OR Renta_Media = 'NaN') AS renta_nula_o_nan,
        COUNT(*) FILTER (WHERE TRY_CAST(Renta_Media AS DOUBLE) IS NOT NULL) AS renta_valida,
        ROUND(MIN(TRY_CAST(Renta_Media AS DOUBLE)), 2) AS renta_min,
        ROUND(MAX(TRY_CAST(Renta_Media AS DOUBLE)), 2) AS renta_max,
        ROUND(AVG(TRY_CAST(Renta_Media AS DOUBLE)), 2) AS renta_media,
        ROUND(MEDIAN(TRY_CAST(Renta_Media AS DOUBLE)), 2) AS renta_mediana
    FROM bronze.mosaic
""").fetchdf()

print("Análisis de la variable Renta_Media:\n")
print(analisis_renta.to_string(index=False))

Análisis de la variable Renta_Media:

 total_cps  renta_nula_o_nan  renta_valida  renta_min  renta_max  renta_media  renta_mediana
      6457               626          5831     2370.0    37743.0     23594.02        24111.0


## 10. Cobertura prevista al cruzar con `dim_cliente`

Esta es una de las preguntas más importantes del notebook: ¿qué porcentaje de los clientes de Selmark podrá ser enriquecido con información MOSAIC? La respuesta determina el alcance real de la línea de geomarketing del TFG.

Para responder se simula el JOIN entre ambas tablas, aplicando el padding necesario al CP de MOSAIC para que coincida con el formato de `dim_cliente`.

In [17]:
# Estimación de cobertura del cruce dim_cliente con mosaic
cobertura = con.execute("""
    WITH mosaic_padded AS (
        SELECT LPAD(CP, 5, '0') AS cp_norm
        FROM bronze.mosaic
    ),
    clientes_con_cp AS (
        SELECT 
            id_cliente,
            codigo_postal_cliente,
            CASE 
                WHEN LENGTH(codigo_postal_cliente) = 5 THEN codigo_postal_cliente
                ELSE NULL
            END AS cp_norm
        FROM bronze.dim_cliente
    )
    SELECT 
        COUNT(*) AS total_clientes,
        COUNT(*) FILTER (WHERE c.cp_norm IS NOT NULL) AS clientes_con_cp_5dig,
        COUNT(*) FILTER (WHERE c.cp_norm IN (SELECT cp_norm FROM mosaic_padded)) AS clientes_con_match_mosaic,
        ROUND(
            COUNT(*) FILTER (WHERE c.cp_norm IN (SELECT cp_norm FROM mosaic_padded)) * 100.0 / COUNT(*), 
            2
        ) AS pct_cobertura_total,
        ROUND(
            COUNT(*) FILTER (WHERE c.cp_norm IN (SELECT cp_norm FROM mosaic_padded)) * 100.0 / 
            NULLIF(COUNT(*) FILTER (WHERE c.cp_norm IS NOT NULL), 0),
            2
        ) AS pct_cobertura_sobre_espanoles
    FROM clientes_con_cp c
""").fetchdf()

print("Cobertura del cruce dim_cliente con mosaic:\n")
print(cobertura.to_string(index=False))

Cobertura del cruce dim_cliente con mosaic:

 total_clientes  clientes_con_cp_5dig  clientes_con_match_mosaic  pct_cobertura_total  pct_cobertura_sobre_espanoles
           3469                  2763                       2088                60.19                          75.57


## 11. Conclusiones del análisis de MOSAIC

### Características generales

- La tabla contiene información sociodemográfica para 6.457 códigos postales españoles, distribuidos entre las 52 provincias del país.
- La estructura combina segmentación fina (50+ segmentos), segmentación agregada (11 grupos A-K), y variables socioeconómicas como la renta media.
- El nivel de análisis recomendado para el TFG es el de **grupo dominante** (`Max_Mosaic_G`), por su balance entre interpretabilidad y granularidad.

### Problemas de calidad detectados

1. **Formato del código postal**: los CPs se han cargado sin ceros iniciales (4 dígitos en lugar de 5). Será necesario aplicar padding en Silver antes del JOIN con `dim_cliente`.
2. **Valores nulos en `Renta_Media`**: una parte de los códigos postales no dispone de renta media estimada. Se documentará el porcentaje y se decidirá su tratamiento (imputación o exclusión).
3. **Tipo de datos inferido como VARCHAR**: todas las columnas se cargaron como texto, incluyendo las numéricas. En Silver se castearán las variables relevantes a sus tipos correctos (DOUBLE para proporciones y renta).

### Decisiones de modelado para la capa Silver

Se construirá una vista Silver simplificada de MOSAIC que conserve únicamente las columnas relevantes para el análisis del TFG:

| Variable Silver | Origen | Descripción |
|---|---|---|
| `codigo_postal_norm` | `LPAD(CP, 5, '0')` | Código postal con 5 dígitos |
| `provincia` | `PROV_INE` | Nombre de la provincia |
| `mosaic_grupo` | `Max_Mosaic_G` | Grupo MOSAIC dominante (A-K) |
| `mosaic_grupo_peso` | `Max_Mosaic2` | Peso del grupo dominante |
| `mosaic_segmento` | `Max_Mosaic` | Segmento fino dominante |
| `mosaic_segmento_peso` | `Max_Mosaic1` | Peso del segmento dominante |
| `renta_media` | `TRY_CAST(Renta_Media)` | Renta media estimada en euros |

Las 50 columnas de proporciones por segmento fino se mantendrán solo en Bronze, disponibles para análisis avanzados si fuera necesario.

### Pendiente de confirmar con el tutor o con Selmark

- Documentación oficial de los grupos MOSAIC (qué representa exactamente cada letra A-K en términos de perfil sociodemográfico).: informe 
- Año de referencia de los datos (¿2023? ¿2024?), relevante para la coherencia temporal del análisis.: 2015

### Próximo notebook

`03_silver_dim_cliente.ipynb` — Construcción de la dimensión de clientes limpia, con normalización geográfica y enriquecimiento con flags analíticos.

## 12. Documentación oficial de los grupos MOSAIC (Experian)

Una vez disponible la documentación oficial de la herramienta MOSAIC España (V.5, Experian), es posible interpretar comercialmente los grupos detectados en la sección anterior. La siguiente tabla resume cada grupo, su perfil sociodemográfico predominante y su lectura desde el punto de vista de Selmark como empresa del sector moda y lencería.

### Resumen de los 11 grupos MOSAIC

| Grupo | Nombre | Perfil predominante | Potencial Selmark |
|---|---|---|---|
| **A** | Élites | Zonas urbanas premium, alto status, formación elevada, sector servicios y directivos | **Alto** — producto premium, marca, campañas de valor |
| **B** | Urbanitas | Centros urbanos densos, perfiles heterogéneos (medios y modestos) | Medio — segmentar por subtipo |
| **C** | Éxito Provincial | Clase media-alta en capitales de provincia, buen nivel residencial | **Alto** — mercado natural fuera de grandes ciudades |
| **D** | Juventud en Expansión | Parejas jóvenes con hijos, nuevas urbanizaciones, hipoteca | **Medio-alto** — moda joven, campañas familiares |
| **E** | Profesionales Maduros | Familias asentadas, status medio, sector terciario | **Alto** — fidelidad y recurrencia |
| **F** | Turismo | Áreas turísticas con actividad comercial y hostelera | Medio — estacionalidad |
| **G** | Industria | Afueras urbanas industriales, trabajadores cualificados | Medio — sensibilidad a precio |
| **H** | Áreas Mixtas | Baja densidad, mezcla industrial-agrícola | Medio-bajo — cobertura territorial |
| **I** | No Cualificados | Familias modestas, trabajos manuales, sensibilidad a precio | Bajo-medio — analizar rentabilidad |
| **J** | Agricultura | Zonas rurales agrícolas/ganaderas | Bajo — cobertura local |
| **K** | Áreas Pasivas | Zonas rurales envejecidas, baja actividad económica | Bajo — solo turismo rural o clientes históricos |

### Comparativa entre la distribución nacional y la observada en MOSAIC

Es importante destacar que los porcentajes recogidos en la documentación oficial reflejan el **peso poblacional nacional**, mientras que el análisis realizado sobre `bronze.mosaic` calcula el **porcentaje de códigos postales**. Estas dos medidas pueden divergir significativamente cuando un grupo concentra mucha población en pocos códigos postales (por ejemplo, los grupos urbanos B y E) o, al contrario, cuando un grupo se reparte entre muchos códigos postales con escasa densidad (grupos rurales como K).

| Grupo | % poblacional nacional (Experian) | % CPs en MOSAIC | Interpretación |
|---|---|---|---|
| K | 9,76 % | 30,53 % | Muchos CPs rurales pequeños |
| H | 15,35 % | 22,49 % | CPs distribuidos territorialmente |
| C | 11,91 % | 10,75 % | Coincidencia razonable |
| B | 13,89 % | 3,48 % | Pocos CPs pero muy densos (urbanos) |
| E | 10,98 % | 2,82 % | Pocos CPs pero muy poblados |
| A | 6,98 % | 4,57 % | Pocos CPs y exclusivos |

**Implicación metodológica**: en el análisis de geomarketing del TFG, el peso comercial de cada grupo no debe medirse únicamente por el número de códigos postales, sino también ponderando por la población residente o, en su defecto, por el volumen de ventas y clientes activos de Selmark en cada zona.

### Variables derivadas a construir en Silver

A partir de la lectura comercial de los grupos, se construirán en la capa Silver las siguientes variables binarias adicionales que faciliten el análisis posterior:

| Variable Silver | Definición | Uso analítico |
|---|---|---|
| `perfil_premium` | 1 si `mosaic_grupo` IN ('A', 'C') | Identificar zonas con alto poder adquisitivo |
| `perfil_familiar_joven` | 1 si `mosaic_grupo` IN ('D', 'E') | Identificar zonas de familias en crecimiento |
| `perfil_turistico` | 1 si `mosaic_grupo` = 'F' o `mosaic_segmento` IN ('C14', 'K44') | Análisis estacional |
| `perfil_rural` | 1 si `mosaic_grupo` IN ('J', 'K') | Cobertura rural |
| `perfil_precio_sensible` | 1 si `mosaic_grupo` IN ('I', 'H', 'K') | Estrategia de precio |

Estas variables, combinadas con `renta_media`, permitirán construir tipologías comerciales claras al final del proyecto.

## 13. Conclusión final del análisis MOSAIC

Tras la incorporación de la documentación oficial de Experian, queda confirmado que la tabla `mosaic` constituye una pieza analítica de gran valor para el TFG. Permite:

1. **Enriquecer cada cliente** con un perfil sociodemográfico territorial sin necesidad de datos personales adicionales.
2. **Interpretar comercialmente** los segmentos resultantes del clustering posterior, dando significado de negocio a los clusters detectados.
3. **Identificar zonas potenciales** para futuras acciones comerciales, cruzando ventas reales con perfiles MOSAIC.
4. **Diferenciar estrategias** según el tipo de zona (premium, familiar, rural, turística, etc.).

La cobertura efectiva del 60% de la cartera de Selmark, aunque parcial, es suficiente para extraer conclusiones representativas, especialmente si se complementa con análisis específicos sobre el subconjunto de clientes españoles con MOSAIC disponible.

El siguiente paso es construir la dimensión de clientes limpia (`silver.dim_cliente`) e ir incorporando progresivamente el resto de tablas hasta llegar a la tabla cliente final (`gold.cliente_360`), donde MOSAIC se integrará como bloque de variables geográfico-socioeconómicas.

---

**Próximo notebook**: `03_silver_dim_cliente.ipynb` — Construcción de la dimensión de clientes limpia, con normalización geográfica y enriquecimiento con flags analíticos.

In [18]:
# ============================================================
# CIERRE DE LA SESIÓN
# ============================================================
# Libera la conexión a DuckDB para evitar bloqueos en otros notebooks

try:
    con.close()
    print("Conexión a DuckDB cerrada correctamente.")
except Exception as e:
    print(f"Aviso al cerrar conexión: {e}")

import gc
gc.collect()

print("\nNotebook finalizado. La base está libre para otros notebooks.")
print("Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.")

Conexión a DuckDB cerrada correctamente.

Notebook finalizado. La base está libre para otros notebooks.
Recomendación: 'Kernel -> Shutdown' antes de abrir el siguiente notebook.
